This involves a take-home task on a toy dataset for you to demonstrate your capability to work on the research project. Please read the instructions below carefully. The task itself is straightforward and should not take more than 2 days to finish. That said, you will have 14 days to finish the task (until 7/16).
1. Your task is to predict the "stance" label of the 5751 tweets in the attached csv file. Specifically, your goal is to infer the stance of a tweet with respect to COVID-19 vaccination. The stance belongs to one of the three categories, "in-favor", "against", "neutral-or-unclear".
2. Data description
    - columns:
        -   "tweet": the text
        -   "label_true": the ground-truth label annotated by human raters.
        -   "label_pred": your predictions. Please fill your predictions into this column.
3. In order to generate the predictions, please use the following model.
    - https://huggingface.co/google/flan-t5-large
4. To feed the tweet into the model, you have to install the HuggingFace and PyTorch package in Python.
    - The instruction to feed the tweets to the model is described here. You should be able to run it on a CPU machine without a GPU.
        - https://huggingface.co/google/flan-t5-large#running-the-model-on-a-cpu
5. Critically, when you feed the tweet into the model, you should embed the tweet in a prompt such that the model can generate the prediction for the stance. One simple prompt you can use is as follows.
> What is the stance of the following tweet with respect to COVID-19 vaccine?  Here is the tweet. "{THE  }"  Please use exactly one word from the following 3 categories to label it: "in-favor", "against", "neutral-or-unclear".
6. Now, please predict the label "label_pred" in the csv file "Q2_20230202_majority.csv". Please keep in mind that the actual dataset consists of around 1 million of tweets, so you want to solve the program programitically.
7. Meanwhile, please push your code to your GitHub in a private repository (don't set it to public!).
8. Fine-tuning is highly encouraged. You may need GPU for fine-tuning. Consider using https://colab.research.google.com/ if you don't have other GPU compute. T4 GPU it provides for free should be enought to fine-tune a FLAN-T5-Large model.
9. After you finished the task, please (a) invite me as a collaborator to the repository. (b) share with me the link to your github repo and (c) send me the csv file with the columns "label_pred" filled with your predictions.
Your attempt will be evaluated based on the three criteria.
(1) The model F1 score on the held-out test set that you did not see.
(2) The readability of the codes, including comments, the use of OOP, etc. Please also include a README file at the repo's root directory.
(3) Whether you use Git control properly. This includes clear git commits etc.

In [ ]:
#!pip install transformers datasets accelerate peft trl

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("Q2_20230202_majority 1.csv")
df.sample(5)

,tweet_id,created_at,tweet,label_majority,month
3082,1.389903e+18,2021-05-05 11:20:07+00:00,they should do this in south africa. they would get all of us vaccinated in 2 days.,in-favor,21-May
828,1.437436e+18,2021-09-13 15:19:44+00:00,imagine calling healthy people sick..if you are not vaccinated you are considered sick? ..noo guys please attach the science that says that and make it make sense after that.,neutral-or-unclear,21-Sep
2162,1.473486e+18,2021-12-22 02:48:14+00:00,the pro-life party cheers for kyle rittenhouse and boos vaccinations.,neutral-or-unclear,21-Dec
3415,1.378018e+18,2021-04-02 16:13:35+00:00,when the whole group message gets vaccinated,in-favor,21-Apr
1462,1.453495e+18,2021-10-27 22:53:54+00:00,ron desantis is a vaccine mandater. .ron desantis is a vaccine mandater. .ron desantis is a vaccine mandater. ..turns out he’s just been playing covid politics.,against,21-Oct


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["tweet", "label_majority"]])
dataset = dataset.train_test_split(test_size=0.15)
print("Dataset loaded and split")

Dataset loaded and split


In [ ]:
from textwrap import dedent

def preprocess(batch):

  prompts = [
  dedent(
    f"""
    Given you're an expert medical journalist, please classify the stance of the following tweet toward the COVID-19 vaccine. [in-favor, against, neutral or unclear]
    What is the stance of the following tweet with respect to COVID-19 vaccination?

    Tweet: {tweet}

    Respond with a detailed rationale and label it as:
    1 for in-favor,
    -1 for against,
    0 neutral-or-unclear,

    Output format:
    Reasoning: [your reasoning]
    Label: [your label]
    """
  )
  for tweet in batch["tweet"]
  ]


  model_inputs = tokenizer(prompts, truncation=True, padding="max_length", max_length=256)

  with tokenizer.as_target_tokenizer():
      labels = tokenizer(batch["label_majority"], truncation=True, padding="max_length", max_length=10)

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs
print("Function laoded")

Function laoded


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
tokenized_ds = dataset.map(preprocess, batched=True)

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

# input_text = "translate English to German: How old are you?"
# input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# outputs = model.generate(input_ids)
# print(tokenizer.decode(outputs[0]))

print("Tokenizer and model loaded!")

Map:   0%|          | 0/3679 [00:00<?, ? examples/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenizer and model loaded!


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)
# During inference:
# inputs = tokenizer("Tweet: I'm happy I got the vaccine.", return_tensors="pt").to(device)
# outputs = model.generate(**inputs)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

@samantha_samantha i'm glad you got the


In [ ]:
from transformers import T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-tweet-stance",
    eval_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_dir='./logs',
    logging_steps=100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer
)
print("Training...")
trainer.train()
print("model trained")

/tmp/ipython-input-19-1419398113.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Training...


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: contactmaanan (contactmaanan-university-of-wisconsin-madison) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [ ]:
df.head()